In [11]:
import sagemaker
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker.tuner import HyperparameterTuner, IntegerParameter
from sagemaker.analytics import HyperparameterTuningJobAnalytics
import boto3

# Function to fetch selected columns from S3
def get_selected_columns():
    bucket_name = "bdp-feature-selection"
    file_key = "data/selected_columns.txt"
    s3_client = boto3.client("s3")
    response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
    file_content = response["Body"].read().decode("utf-8")
    return file_content.splitlines()

# Initialize SageMaker session and role
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
image_uri = sagemaker.image_uris.retrieve("randomcutforest", region=sagemaker_session.boto_region_name)

# S3 paths
train_data_path = "s3://bdp-recordio/train/"
test_data_path = "s3://bdp-test-data/scaled/test_data_numerical_label.csv"
output_path = "s3://bdp-models/rcf/"

# Get selected columns and calculate feature dimension
selected_columns = get_selected_columns()
feature_dim = len(selected_columns)

# Define RCF Estimator
rcf = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.c5.xlarge",
    output_path=output_path,
    sagemaker_session=sagemaker_session,
    input_mode="Pipe"
)

rcf.set_hyperparameters(
    feature_dim=feature_dim
)

# Define hyperparameter ranges for tuning
hyperparameter_ranges = {
    "num_samples_per_tree": IntegerParameter(100, 500),  # Range of samples per tree
    "num_trees": IntegerParameter(128, 1000),            # Range of number of trees
}

# Objective metric for tuning (maximize F1 score)
objective_metric_name = "test:f1"
objective_type = "Maximize"

# Define inputs
train_input = TrainingInput(
    train_data_path,
    content_type="application/x-recordio-protobuf",
    distribution="ShardedByS3Key",
)

test_input = TrainingInput(
    test_data_path,
    content_type="text/csv;label_size=1",
    distribution="FullyReplicated",
)

# Configure the hyperparameter tuner
tuner = HyperparameterTuner(
    estimator=rcf,
    objective_metric_name=objective_metric_name,
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=10,                 # Total number of HPO jobs
    max_parallel_jobs=2,         # Number of parallel jobs
)

# Start hyperparameter tuning
tuner.fit({"train": train_input, "test": test_input}, wait=True)

# Analyze tuning results
tuning_job_name = tuner.latest_tuning_job.name
print(f"Tuning job name: {tuning_job_name}")

# Analyze tuning results
tuning_analytics = HyperparameterTuningJobAnalytics(tuning_job_name)
results_df = tuning_analytics.dataframe()

if not results_df.empty:
    print("Hyperparameter Tuning Results:")
    print(results_df)
    results_df.to_csv(f"{tuning_job_name}_results.csv", index=False)
    print(f"Results saved to {tuning_job_name}_results.csv")
else:
    print("No tuning results were returned.")


[01/24/25 20:32:53] INFO     Same images used for training and inference. Defaulting to image     ]8;id=914493;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=114432;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#391\391]8;;\
                             scope: inference.                                                                     

                    INFO     Ignoring unnecessary instance type: None.                            ]8;id=938855;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=392746;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#528\528]8;;\

                    WARNING  No finished training job found associated with this estimator.       ]8;id=891730;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py\estimator.py]8;;\:]8;id=172796;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py#1914\1914]8;;\
                             Please make sure this estimator is only used for building workflow                    
                             config                                                                                

                    WARNING  No finished training job found associated with this estimator.       ]8;id=876368;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py\estimator.py]8;;\:]8;id=305323;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py#1914\1914]8;;\
                             Please make sure this estimator is only used for building workflow                    
                             config                                                                                

                    INFO     Creating hyperparameter tuning job with name:                          ]8;id=132279;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=232801;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#3383\3383]8;;\
                             randomcutforest-250124-2032                                                           

..............................................................................................................................................................................................................................!
Tuning job name: randomcutforest-250124-2032
Hyperparameter Tuning Results:
   num_samples_per_tree  num_trees                           TrainingJobName  \
0                 102.0      578.0  randomcutforest-250124-2032-010-36a01df4   
1                 382.0      152.0  randomcutforest-250124-2032-009-d5f244c4   
2                 101.0      367.0  randomcutforest-250124-2032-008-ea3d7aed   
3                 155.0      942.0  randomcutforest-250124-2032-007-2e3230a7   
4                 460.0      135.0  randomcutforest-250124-2032-006-e8f85576   
5                 374.0      339.0  randomcutforest-250124-2032-005-789219fb   
6                 399.0      988.0  randomcutforest-250124-2032-004-e0f1330d   
7                 370.0      985.0  randomcutforest-250124-2

In [ ]:
hyperparameter_ranges = {
    "num_samples_per_tree": IntegerParameter(1, 2048),
    "num_trees": IntegerParameter(50, 200)
}

objective_metric_name = "test:f1"
objective_type = "Maximize"

tuner = HyperparameterTuner(
    estimator=rcf,
    objective_metric_name=objective_metric_name,
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=10,  # Total number of HPO jobs
    max_parallel_jobs=2,  # Number of parallel jobs
)

# Start the HPO job with both training and test datasets
tuner.fit({"train": train_input, "test": test_input}, wait=True)

# Analyze the HPO job results
tuning_job_name = tuner.latest_tuning_job.name
print(f"Tuning Job Name: {tuning_job_name}")

# Retrieve the best model hyperparameters
tuning_job_analytics = HyperparameterTuningJobAnalytics(tuning_job_name)
if not tuning_job_analytics.dataframe().empty:
    print("Hyperparameter Tuning Results:")
    print(tuning_job_analytics.dataframe())

# Retrieve the best model artifacts
best_model_job_name = tuner.best_training_job()
print(f"Best Model Training Job Name: {best_model_job_name}")
